# Preparação do Dataset Analítico

Este notebook tem como objetivo consolidar as bases da camada Gold em um único dataset analítico para as etapas de Análise Exploratória de Dados (EDA) e Machine Learning.

Serão utilizados dois conjuntos principais:

- `gold_alunos.parquet`: dados em nível individual;
- `gold_municipal.parquet`: dados agregados em nível municipal.

Ao longo do notebook serão realizadas:

- análise estrutural das bases;
- validação de granularidade;
- análise de qualidade;
- identificação das chaves de integração;
- avaliação da relevância das variáveis;
- identificação de possíveis riscos de data leakage;
- seleção das variáveis utilizadas no dataset consolidado;
- integração das bases;
- validação do dataset final.

O objetivo é produzir uma base única, consistente e documentada para as próximas etapas do projeto.

## 1. Importação das bibliotecas

Inicialmente são carregadas as bibliotecas utilizadas para manipulação, validação e análise estrutural dos dados.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Configurações gerais

São definidas configurações de visualização do Pandas para facilitar a inspeção dos datasets durante o processo de preparação.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 3. Definição dos caminhos

Os datasets utilizados neste notebook estão armazenados na camada Gold do projeto.

In [3]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data"
GOLD_DIR = DATA_DIR / "gold"

ALUNOS_PATH = GOLD_DIR / "gold_alunos.parquet"
MUNICIPAL_PATH = GOLD_DIR / "gold_municipal.parquet"

## 4. Carregamento dos datasets

São carregadas as duas principais bases analíticas disponíveis.

A base de alunos possui granularidade individual, enquanto a base municipal contém informações agregadas relacionadas aos indicadores de alfabetização e metas educacionais.

In [4]:
df_alunos = pd.read_parquet(ALUNOS_PATH)
df_municipal = pd.read_parquet(MUNICIPAL_PATH)

print(f"Alunos: {df_alunos.shape[0]:,} linhas | {df_alunos.shape[1]} colunas")
print(f"Municipal: {df_municipal.shape[0]:,} linhas | {df_municipal.shape[1]} colunas")

Alunos: 57,782 linhas | 16 colunas
Municipal: 23,995 linhas | 51 colunas


## 5. Visão inicial dos dados

Antes de realizar qualquer seleção ou integração, é importante conhecer a estrutura das duas bases e compreender quais informações estão disponíveis em cada uma.

In [5]:
display(df_alunos.head())

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,sigla_uf,meta_municipio_ano,meta_uf_ano
0,2023,1302504,Manacapuru,60000951,13015851,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN
1,2023,1302603,Manaus,60000963,13030738,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN
2,2023,1300631,Beruri,60001351,13003982,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN
3,2023,1711506,Jaú do Tocantins,60004115,17012510,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN
4,2023,2100709,Anajatuba,60004434,21012344,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,<NA>,NaN,NaN


In [6]:
display(df_municipal.head())

,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,ano_meta_municipio,taxa_alfabetizacao_meta_municipio,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,ano_meta_uf,sigla_uf,taxa_alfabetizacao_meta_uf,meta_alfabetizacao_2024_meta_uf,meta_alfabetizacao_2025_meta_uf,meta_alfabetizacao_2026_meta_uf,meta_alfabetizacao_2027_meta_uf,meta_alfabetizacao_2028_meta_uf,meta_alfabetizacao_2029_meta_uf,meta_alfabetizacao_2030_meta_uf,percentual_participacao_meta_uf,ano_meta_brasil,taxa_alfabetizacao_meta_brasil,meta_alfabetizacao_2024_meta_brasil,meta_alfabetizacao_2025_meta_brasil,meta_alfabetizacao_2026_meta_brasil,meta_alfabetizacao_2027_meta_brasil,meta_alfabetizacao_2028_meta_brasil,meta_alfabetizacao_2029_meta_brasil,meta_alfabetizacao_2030_meta_brasil,percentual_participacao_meta_brasil,meta_municipio_ano,meta_uf_ano,gap_meta,atingiu_meta
0,2023,1100031,2,Municipal,69.10,767.88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00",75.88,70.85,72.53,74.15,75.71,77.21,78.64,80.00,4.00,92.59,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2023,1100072,2,Municipal,58.20,747.89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,023.00",58.20,61.82,65.31,68.64,71.79,74.74,77.48,80.00,2.00,92.25,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2023,1100189,2,Privada,69.73,762.41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2023,1101609,2,Municipal,50.70,745.68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00",78.73,55.53,60.25,64.80,69.09,73.07,76.71,80.00,4.00,91.20,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2023,1101807,2,Municipal,55.69,752.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00",44.06,59.72,63.63,67.37,70.89,74.18,77.22,80.00,1.00,88.64,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [7]:
print("Colunas - Alunos")
print(df_alunos.columns.tolist())

print("\nColunas - Municipal")
print(df_municipal.columns.tolist())

Colunas - Alunos
['ano', 'id_municipio', 'id_municipio_nome', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado', 'proficiencia', 'peso_aluno', 'sigla_uf', 'meta_municipio_ano', 'meta_uf_ano']

Colunas - Municipal
['ano', 'id_municipio', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', 'ano_meta_municipio', 'taxa_alfabetizacao_meta_municipio', 'meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030', 'nivel_alfabetizacao', 'percentual_participacao', 'ano_meta_uf', 'sigla_uf', 'taxa_alfabetizacao_meta_uf', 'meta_alfabetizacao_2024_meta_uf', 'meta_alfabetizac

## 6. Análise estrutural

Nesta etapa será realizada uma análise estrutural dos datasets, considerando:

- quantidade de registros;
- quantidade de colunas;
- tipos dos dados;
- valores ausentes;
- cardinalidade;
- duplicidades.

O objetivo não é realizar ainda a análise exploratória completa, mas verificar se as bases estão adequadas para integração e construção do dataset analítico.

In [8]:
def resumo_estrutura(df):
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "pct_nulos": (df.isna().mean() * 100).round(2),
        "valores_unicos": df.nunique(dropna=True)
    })

In [9]:
display(
    resumo_estrutura(df_alunos)
)

,tipo,nulos,pct_nulos,valores_unicos
ano,int64,0,0.00,2
id_municipio,string,0,0.00,4591
id_municipio_nome,string,0,0.00,4397
id_escola,string,0,0.00,24346
id_aluno,string,0,0.00,57449
caderno,int64,0,0.00,4
serie,string,0,0.00,1
rede,string,0,0.00,2
presenca,string,0,0.00,2
preenchimento_caderno,string,0,0.00,2


In [10]:
display(
    resumo_estrutura(df_municipal)
)

,tipo,nulos,pct_nulos,valores_unicos
ano,int64,0,0.00,2
id_municipio,string,0,0.00,5550
serie,int64,0,0.00,1
rede,string,0,0.00,4
taxa_alfabetizacao,float64,0,0.00,6161
media_portugues,float64,0,0.00,13321
proporcao_aluno_nivel_0,float64,11547,48.12,943
proporcao_aluno_nivel_1,float64,11547,48.12,1374
proporcao_aluno_nivel_2,float64,11547,48.12,1922
proporcao_aluno_nivel_3,float64,11547,48.12,2289


## 7. Diagnóstico da cobertura dos dados

A análise estrutural identificou uma quantidade elevada de valores ausentes em algumas variáveis contextuais, especialmente nas informações relacionadas às metas municipais, estaduais e nacionais.

Antes da seleção das variáveis e da integração das bases, será investigada a distribuição temporal dos dados para verificar se a ausência dessas informações está relacionada aos anos disponíveis no dataset ou a problemas de integração entre as fontes.

In [11]:
print("ANOS - BASE DE ALUNOS")
display(
    df_alunos["ano"]
    .value_counts()
    .sort_index()
)

print("\nANOS - BASE MUNICIPAL")
display(
    df_municipal["ano"]
    .value_counts()
    .sort_index()
)

ANOS - BASE DE ALUNOS


ano
2023    28295
2024    29487
Name: count, dtype: int64


ANOS - BASE MUNICIPAL


ano
2023    11547
2024    12448
Name: count, dtype: int64

In [12]:
cobertura_municipal = (
    df_municipal
    .groupby("ano")
    .agg(
        registros=("id_municipio", "size"),

        pct_taxa_alfabetizacao=(
            "taxa_alfabetizacao",
            lambda x: x.notna().mean() * 100
        ),

        pct_meta_municipio=(
            "meta_municipio_ano",
            lambda x: x.notna().mean() * 100
        ),

        pct_meta_uf=(
            "meta_uf_ano",
            lambda x: x.notna().mean() * 100
        ),

        pct_sigla_uf=(
            "sigla_uf",
            lambda x: x.notna().mean() * 100
        )
    )
    .round(2)
)

display(cobertura_municipal)

,registros,pct_taxa_alfabetizacao,pct_meta_municipio,pct_meta_uf,pct_sigla_uf
ano,,,,,
2023,11547,100.00,0.00,0.00,0.00
2024,12448,100.00,42.03,42.03,0.00


### 7.1 Recuperação da Unidade Federativa

A variável `sigla_uf` apresentou 100% de valores ausentes na base municipal.

Como o identificador do município segue a codificação oficial do IBGE, os dois primeiros dígitos de `id_municipio` identificam a Unidade Federativa.

Dessa forma, a UF pode ser reconstruída diretamente a partir do código municipal, sem necessidade de imputação estatística.

In [13]:
mapa_uf = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR",
    "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN",
    "25": "PB", "26": "PE", "27": "AL", "28": "SE",
    "29": "BA", "31": "MG", "32": "ES", "33": "RJ",
    "35": "SP", "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF"
}

df_municipal["sigla_uf"] = (
    df_municipal["id_municipio"]
    .astype("string")
    .str.zfill(7)
    .str[:2]
    .map(mapa_uf)
)

In [14]:
print("Valores ausentes em sigla_uf:")
print(df_municipal["sigla_uf"].isna().sum())

print("\nQuantidade de UFs:")
print(df_municipal["sigla_uf"].nunique())

print("\nDistribuição:")
display(
    df_municipal["sigla_uf"]
    .value_counts()
    .sort_index()
)

Valores ausentes em sigla_uf:
0

Quantidade de UFs:
26

Distribuição:


sigla_uf
AC      55
AL     423
AM     357
AP      77
BA    2061
CE     757
DF       2
ES     371
GO     985
MA     875
MG    3877
MS     349
MT     623
PA     598
PB     930
PE     746
PI     898
PR    1604
RJ     370
RN     816
RO     226
RS    2483
SC    1450
SE     344
SP    2162
TO     556
Name: count, dtype: int64

### 7.2 Validação da cobertura territorial

Após reconstruir a Unidade Federativa a partir do código do município, foram identificadas 26 UFs distintas.

Como o Brasil possui 27 unidades federativas, será verificado se a ausência de alguma UF é consequência da cobertura original dos dados ou de inconsistências nos identificadores municipais.

In [15]:
ufs_esperadas = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
}

ufs_presentes = set(
    df_municipal["sigla_uf"]
    .dropna()
    .unique()
)

ufs_ausentes = ufs_esperadas - ufs_presentes

print("UFs ausentes:")
print(ufs_ausentes)

UFs ausentes:
{'RR'}


In [16]:
df_municipal[
    df_municipal["id_municipio"]
    .astype("string")
    .str.zfill(7)
    .str.startswith("14")
][
    ["id_municipio", "ano", "rede"]
].head()

,id_municipio,ano,rede


A validação territorial identificou ausência de registros para o estado de Roraima (RR).

Como não foram encontrados municípios com prefixo IBGE `14`, conclui-se que a ausência é decorrente da cobertura original da base e não de falha na reconstrução da variável `sigla_uf`.

Essa limitação será considerada nas análises territoriais posteriores.

In [17]:
cobertura_meta_2024 = (
    df_municipal[df_municipal["ano"] == 2024]
    .groupby("rede")
    .agg(
        registros=("id_municipio", "size"),
        pct_meta_municipio=(
            "meta_municipio_ano",
            lambda x: x.notna().mean() * 100
        ),
        pct_meta_uf=(
            "meta_uf_ano",
            lambda x: x.notna().mean() * 100
        )
    )
    .round(2)
)

display(cobertura_meta_2024)

,registros,pct_meta_municipio,pct_meta_uf
rede,,,
Estadual,1086,0.00,0.00
Federal,398,0.00,0.00
Municipal,5448,96.04,96.04
Privada,5516,0.00,0.00


In [18]:
display(
    df_municipal.loc[
        df_municipal["ano"].eq(2024),
        ["rede", "meta_municipio_ano", "meta_uf_ano"]
    ]
    .groupby("rede")
    .size()
)

rede
Estadual     1086
Federal       398
Municipal    5448
Privada      5516
dtype: int64

### 7.3 Cobertura das metas por rede de ensino

A análise da cobertura das metas em 2024 mostrou que as informações estão concentradas na rede Municipal.

Enquanto aproximadamente 96% dos registros municipais possuem metas associadas, as redes Estadual, Federal e Privada não apresentam cobertura para essas variáveis.

Dessa forma, os valores ausentes observados nas metas não devem ser interpretados apenas como problema de qualidade dos dados, mas também como consequência da própria disponibilidade das metas por rede de ensino.

Essa característica deverá ser considerada na seleção das variáveis utilizadas no dataset analítico final.

In [19]:
distribuicao_rede_alunos = (
    df_alunos["rede"]
    .value_counts()
    .to_frame("quantidade")
)

distribuicao_rede_alunos["percentual"] = (
    distribuicao_rede_alunos["quantidade"]
    / len(df_alunos)
    * 100
).round(2)

display(distribuicao_rede_alunos)

,quantidade,percentual
rede,,
Municipal,51455,89.05
Estadual,6327,10.95


In [20]:
display(
    pd.crosstab(
        df_alunos["rede"],
        df_alunos["ano"]
    )
)

ano,2023,2024
rede,,
Estadual,2528,3799
Municipal,25767,25688


### 7.4 Distribuição dos alunos por rede de ensino

A base individual é composta predominantemente por alunos da rede Municipal, que representa aproximadamente 89% dos registros. A rede Estadual corresponde aos cerca de 11% restantes.

Essa distribuição é relevante para a utilização das metas educacionais, uma vez que a cobertura das metas municipais está concentrada na rede Municipal.

Dessa forma, as metas podem contribuir para o contexto educacional da maior parte dos alunos, mas sua ausência na rede Estadual deverá ser considerada durante a preparação e modelagem dos dados.

#### 7.5.1 Padronização das chaves de integração

A tentativa inicial de integração revelou uma incompatibilidade entre os tipos das chaves utilizadas no join.

Em especial, a variável `serie` possui representações diferentes nas duas bases:

- na base individual, a série é representada de forma textual;
- na base municipal, a série é representada numericamente.

Para preservar as informações originais, será criada uma chave auxiliar padronizada exclusivamente para a integração.

In [21]:
print("ALUNOS")
print(df_alunos["serie"].dtype)
print(df_alunos["serie"].unique())

print("\nMUNICIPAL")
print(df_municipal["serie"].dtype)
print(df_municipal["serie"].unique())

ALUNOS
string
<ArrowStringArray>
['2° ano do Ensino Fundamental']
Length: 1, dtype: string

MUNICIPAL
int64
[2]


In [22]:
df_alunos["serie_join"] = (
    df_alunos["serie"]
    .str.extract(r"(\d+)", expand=False)
    .astype("Int64")
)

df_municipal["serie_join"] = (
    df_municipal["serie"]
    .astype("Int64")
)

In [23]:
print("Série - alunos:")
print(df_alunos["serie_join"].value_counts(dropna=False))

print("\nSérie - municipal:")
print(df_municipal["serie_join"].value_counts(dropna=False))

Série - alunos:
serie_join
2    57782
Name: count, dtype: Int64

Série - municipal:
serie_join
2    23995
Name: count, dtype: Int64


In [24]:
for df in [df_alunos, df_municipal]:
    df["id_municipio"] = df["id_municipio"].astype("string")
    df["ano"] = df["ano"].astype("Int64")
    df["rede"] = df["rede"].astype("string").str.strip()

In [25]:
chaves_join = [
    "id_municipio",
    "ano",
    "serie_join",
    "rede"
]



In [26]:
print(df_alunos["serie"].unique())
print(df_municipal["serie"].unique())

<ArrowStringArray>
['2° ano do Ensino Fundamental']
Length: 1, dtype: string
[2]


In [27]:
print(df_alunos[chaves_join].dtypes)
print()
print(df_municipal[chaves_join].dtypes)

id_municipio    string
ano              Int64
serie_join       Int64
rede            string
dtype: object

id_municipio    string
ano              Int64
serie_join       Int64
rede            string
dtype: object


### 7.5.2 Integração entre alunos e contexto municipal

Após a padronização das chaves, as duas bases passam a possuir tipos compatíveis para `id_municipio`, `ano`, `serie` e `rede`.

A integração será realizada utilizando uma relação `many-to-one`, pois vários alunos podem pertencer ao mesmo contexto municipal, enquanto cada combinação de município, ano, série e rede deve possuir apenas um registro na base municipal.

In [28]:
contexto_meta = df_municipal[
    chaves_join
    + [
        "meta_municipio_ano",
        "meta_uf_ano"
    ]
].copy()

In [29]:
df_teste_meta = df_alunos.merge(
    contexto_meta,
    on=chaves_join,
    how="left",
    validate="many_to_one",
    suffixes=("", "_municipal")
)

print("Antes do join:", df_alunos.shape)
print("Depois do join:", df_teste_meta.shape)

Antes do join: (57782, 17)
Depois do join: (57782, 19)


In [30]:
cobertura_meta_alunos = pd.DataFrame({
    "nulos": df_teste_meta[
        [
            "meta_municipio_ano_municipal",
            "meta_uf_ano_municipal"
        ]
    ].isna().sum(),

    "pct_preenchido": (
        df_teste_meta[
            [
                "meta_municipio_ano_municipal",
                "meta_uf_ano_municipal"
            ]
        ]
        .notna()
        .mean()
        .mul(100)
        .round(2)
    )
})

display(cobertura_meta_alunos)

,nulos,pct_preenchido
meta_municipio_ano_municipal,33039,42.82
meta_uf_ano_municipal,33039,42.82


In [31]:
cobertura_meta_alunos_rede = (
    df_teste_meta
    .groupby(["ano", "rede"])
    .agg(
        alunos=("id_aluno", "size"),

        pct_meta_municipio=(
            "meta_municipio_ano_municipal",
            lambda x: x.notna().mean() * 100
        ),

        pct_meta_uf=(
            "meta_uf_ano_municipal",
            lambda x: x.notna().mean() * 100
        )
    )
    .round(2)
)

display(cobertura_meta_alunos_rede)

alunos  pct_meta_municipio  pct_meta_uf
ano  rede                                              
2023 Estadual     2528                0.00         0.00
     Municipal   25767                0.00         0.00
2024 Estadual     3799                0.00         0.00
     Municipal   25688               96.32        96.32

### 7.6 Avaliação da utilidade das metas

A simulação da integração mostrou que as variáveis de meta possuem cobertura global de aproximadamente 42,8% na base individual.

Ao analisar a cobertura por ano e rede, observa-se que:

- não há metas associadas aos registros de 2023;
- não há metas associadas à rede Estadual;
- em 2024, aproximadamente 96% dos alunos da rede Municipal possuem metas disponíveis.

Dessa forma, as metas permanecem relevantes para análises contextuais, principalmente para a rede Municipal em 2024, mas não serão tratadas como variáveis de cobertura completa no dataset final.

A ausência de valores nessas variáveis representa, em grande parte, indisponibilidade estrutural da informação e não necessariamente problema de qualidade dos dados.

### 7.7 Avaliação do percentual de participação

A variável `percentual_participacao` pode representar um contexto relevante da avaliação, porém apresenta quantidade significativa de valores ausentes.

Antes de incluí-la no dataset consolidado, será analisada sua cobertura por ano e rede de ensino.

In [32]:
cobertura_participacao = (
    df_municipal
    .groupby(["ano", "rede"])
    .agg(
        registros=("id_municipio", "size"),
        pct_participacao=(
            "percentual_participacao",
            lambda x: x.notna().mean() * 100
        )
    )
    .round(2)
)

display(cobertura_participacao)

registros  pct_participacao
ano  rede                                  
2023 Estadual        1149              0.00
     Municipal       5448             96.75
     Privada         4950              0.00
2024 Estadual        1086              0.00
     Federal          398              0.00
     Municipal       5448             97.10
     Privada         5516              0.00

### 7.8 Conclusão sobre o percentual de participação

A variável `percentual_participacao` apresentou alta cobertura para a rede Municipal, com aproximadamente 97% dos registros preenchidos em 2023 e 2024.

Nas demais redes de ensino, entretanto, a variável não possui valores disponíveis.

Dessa forma, o percentual de participação será mantido no dataset analítico por seu potencial valor contextual e exploratório, mas sua utilização como feature nos modelos de Machine Learning deverá ser avaliada considerando sua cobertura restrita à rede Municipal.

In [33]:
colunas_municipais_final = [
    "id_municipio",
    "ano",
    "serie_join",
    "rede",
    "sigla_uf",
    "meta_municipio_ano",
    "meta_uf_ano",
    "percentual_participacao"
]

In [34]:
colunas_leakage = [
    "taxa_alfabetizacao",
    "media_portugues",
    "nivel_alfabetizacao",
    "gap_meta",
    "atingiu_meta"
] + [
    col
    for col in df_municipal.columns
    if col.startswith("proporcao_aluno_nivel_")
]

## 8. Seleção definitiva das variáveis municipais

Após a análise de cobertura, granularidade e consistência das variáveis municipais, é realizada a seleção das colunas que serão integradas à base individual.

A seleção considera três critérios principais:

- relevância para o contexto educacional e territorial;
- disponibilidade razoável nos dados;
- risco de data leakage.

Variáveis diretamente relacionadas ao resultado observado de alfabetização não serão utilizadas como features preditivas, mesmo que permaneçam disponíveis para análises exploratórias.

In [35]:
colunas_municipais_final = [
    "id_municipio",
    "ano",
    "serie_join",
    "rede",
    "sigla_uf",
    "meta_municipio_ano",
    "meta_uf_ano",
    "percentual_participacao"
]

In [36]:
colunas_eda_apenas = [
    "taxa_alfabetizacao",
    "media_portugues",
    "nivel_alfabetizacao",
    "gap_meta",
    "atingiu_meta"
]

In [37]:
colunas_leakage = [
    "taxa_alfabetizacao",
    "media_portugues",
    "nivel_alfabetizacao",
    "gap_meta",
    "atingiu_meta"
]

colunas_leakage += [
    col
    for col in df_municipal.columns
    if col.startswith("proporcao_aluno_nivel_")
]

In [38]:
resumo_selecao = pd.DataFrame({
    "grupo": [
        "Dataset final",
        "EDA apenas",
        "Possível leakage"
    ],
    "quantidade_colunas": [
        len(colunas_municipais_final),
        len(colunas_eda_apenas),
        len(colunas_leakage)
    ]
})

display(resumo_selecao)

,grupo,quantidade_colunas
0,Dataset final,8
1,EDA apenas,5
2,Possível leakage,14


A seleção acima evita que o dataset final seja composto por variáveis redundantes ou diretamente derivadas do desempenho educacional observado.

Dessa forma, o dataset consolidado mantém informações contextuais úteis para análise e modelagem, enquanto indicadores de desempenho são preservados apenas como referência analítica.

## 9. Construção do dataset consolidado

Após a seleção das variáveis relevantes, será realizada a integração entre a base individual de alunos e o contexto municipal.

A integração utilizará as seguintes chaves:

- `id_municipio`;
- `ano`;
- `serie`;
- `rede`.

A relação esperada é do tipo `many-to-one`, pois vários alunos podem pertencer ao mesmo contexto municipal, enquanto cada combinação de município, ano, série e rede deve possuir apenas um registro na base municipal.

O objetivo desta etapa é gerar um dataset único, preservando a granularidade individual dos alunos e adicionando informações contextuais relevantes para as etapas de EDA e Machine Learning.

In [40]:
df_contexto_final = (
    df_municipal[
        colunas_municipais_final
    ]
    .copy()
)

In [41]:
print("Shape do contexto municipal:", df_contexto_final.shape)

print(
    "Duplicidades na chave:",
    df_contexto_final.duplicated(
        subset=[
            "id_municipio",
            "ano",
            "serie_join",
            "rede"
        ]
    ).sum()
)

Shape do contexto municipal: (23995, 8)
Duplicidades na chave: 0


In [42]:
colunas_contexto_antigas = [
    "sigla_uf",
    "meta_municipio_ano",
    "meta_uf_ano"
]

df_alunos_base = df_alunos.drop(
    columns=[
        col
        for col in colunas_contexto_antigas
        if col in df_alunos.columns
    ]
).copy()

In [43]:
df_final = df_alunos_base.merge(
    df_contexto_final,
    on=[
        "id_municipio",
        "ano",
        "serie_join",
        "rede"
    ],
    how="left",
    validate="many_to_one"
)

In [44]:
print("Base de alunos antes:", df_alunos_base.shape)
print("Dataset após o join:", df_final.shape)

print(
    "Diferença de registros:",
    len(df_final) - len(df_alunos_base)
)

Base de alunos antes: (57782, 14)
Dataset após o join: (57782, 18)
Diferença de registros: 0


## 10. Validação do dataset consolidado

Após a integração entre a base individual e o contexto municipal, será realizada uma validação final da qualidade do dataset consolidado.

Nesta etapa serão verificados:

- quantidade de registros e colunas;
- duplicidades;
- valores ausentes;
- cardinalidade;
- consistência do target;
- cobertura das variáveis contextuais.

O objetivo é garantir que o dataset esteja adequado para ser utilizado nas próximas etapas de Análise Exploratória de Dados e Machine Learning.

In [45]:
print("Shape final:", df_final.shape)
print("Duplicatas:", df_final.duplicated().sum())

Shape final: (57782, 18)
Duplicatas: 0


In [46]:
resumo_final = pd.DataFrame({
    "tipo": df_final.dtypes.astype(str),
    "nulos": df_final.isna().sum(),
    "pct_nulos": (df_final.isna().mean() * 100).round(2),
    "valores_unicos": df_final.nunique(dropna=True)
})

display(resumo_final)

,tipo,nulos,pct_nulos,valores_unicos
ano,Int64,0,0.00,2
id_municipio,string,0,0.00,4591
id_municipio_nome,string,0,0.00,4397
id_escola,string,0,0.00,24346
id_aluno,string,0,0.00,57449
caderno,int64,0,0.00,4
serie,string,0,0.00,1
rede,string,0,0.00,2
presenca,string,0,0.00,2
preenchimento_caderno,string,0,0.00,2


In [47]:
display(df_final.head())

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,serie_join,sigla_uf,meta_municipio_ano,meta_uf_ano,percentual_participacao
0,2023,1302504,Manacapuru,60000951,13015851,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,2,AM,NaN,NaN,87.06
1,2023,1302603,Manaus,60000963,13030738,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,2,AM,NaN,NaN,71.33
2,2023,1300631,Beruri,60001351,13003982,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,2,AM,NaN,NaN,NaN
3,2023,1711506,Jaú do Tocantins,60004115,17012510,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,2,TO,NaN,NaN,82.98
4,2023,2100709,Anajatuba,60004434,21012344,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,2,MA,NaN,NaN,91.54


### 10.1 Remoção de coluna auxiliar

A variável `serie_join` foi criada exclusivamente para padronizar a chave de integração entre as bases.

Como a variável original `serie` foi preservada no dataset consolidado, a coluna auxiliar será removida antes da geração do arquivo final.

In [48]:
df_final = df_final.drop(
    columns=["serie_join"]
)

### 10.2 Validação da variável alvo

A variável `alfabetizado` será utilizada como target na etapa de Machine Learning.

Antes de salvar o dataset analítico, será verificada sua distribuição e a existência de valores ausentes.

In [49]:
print("Valores ausentes no target:")
print(df_final["alfabetizado"].isna().sum())

display(
    df_final["alfabetizado"]
    .value_counts()
    .to_frame("quantidade")
)

Valores ausentes no target:
0


,quantidade
alfabetizado,
Não,29535
Sim,28247


### 10.3 Variáveis que exigem atenção na modelagem

Algumas variáveis são mantidas no dataset analítico para permitir investigação durante a EDA, mas não serão necessariamente utilizadas diretamente como features do modelo.

Em especial:

- `id_aluno`: identificador único, sem valor preditivo direto;
- `id_escola`: identificador de alta cardinalidade;
- `id_municipio`: utilizado para contexto territorial, mas exige avaliação antes do uso como feature;
- `proficiencia`: possui forte relação com a classificação de alfabetização e pode representar data leakage;
- `presenca`: apresenta associação direta com o target e será analisada antes da modelagem;
- `meta_municipio_ano` e `meta_uf_ano`: possuem cobertura parcial;
- `percentual_participacao`: possui cobertura principalmente associada à rede Municipal.

As decisões finais de seleção, imputação e transformação das features serão realizadas após a Análise Exploratória dos Dados e dentro da pipeline de Machine Learning.

## 11. Persistência do dataset analítico

Após as validações estruturais e a integração das informações individuais e municipais, o dataset consolidado é persistido em formato Parquet.

Esse arquivo será utilizado como fonte principal para as próximas etapas do projeto:

- Análise Exploratória de Dados;
- definição das features;
- treinamento dos modelos;
- avaliação;
- interpretabilidade.

In [50]:
DATASET_FINAL_PATH = (
    GOLD_DIR / "gold_dataset_analitico.parquet"
)

df_final.to_parquet(
    DATASET_FINAL_PATH,
    index=False
)

print("Dataset salvo em:")
print(DATASET_FINAL_PATH)

print("\nShape final:")
print(df_final.shape)

Dataset salvo em:
..\data\gold\gold_dataset_analitico.parquet

Shape final:
(57782, 17)


In [51]:
df_final.head()

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,sigla_uf,meta_municipio_ano,meta_uf_ano,percentual_participacao
0,2023,1302504,Manacapuru,60000951,13015851,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,AM,NaN,NaN,87.06
1,2023,1302603,Manaus,60000963,13030738,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,AM,NaN,NaN,71.33
2,2023,1300631,Beruri,60001351,13003982,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,AM,NaN,NaN,NaN
3,2023,1711506,Jaú do Tocantins,60004115,17012510,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,TO,NaN,NaN,82.98
4,2023,2100709,Anajatuba,60004434,21012344,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,MA,NaN,NaN,91.54


In [52]:
df_final.columns.tolist()

['ano',
 'id_municipio',
 'id_municipio_nome',
 'id_escola',
 'id_aluno',
 'caderno',
 'serie',
 'rede',
 'presenca',
 'preenchimento_caderno',
 'alfabetizado',
 'proficiencia',
 'peso_aluno',
 'sigla_uf',
 'meta_municipio_ano',
 'meta_uf_ano',
 'percentual_participacao']